# Transportadores · datos originales, YAN tratado e inicio theta_full

Se rehace el MISMO notebook08 conforme a la última instrucción. Fuente química LAB004–009: data/original/natural/lab004_012/mosto_natural_original.xlsx, SIN las tres correcciones ni las seis exclusiones de biomasa de la versión depurada. Se normaliza en memoria con las conversiones históricas, conservando todas las biomasas originales disponibles.

**Sólo tratamiento de YAN:** valores<=20mg/L a0; primer agotamiento después del último pulso; LAB006 agotamiento manual122h y muestra122h=0. Se excluyen observaciones YAN con t>agotamiento+12h. NO se aplican cortes tardíos de biomasa: la última instrucción conserva tratamientos de YAN y pide datos sin depurar. No se modifica el resto de las variables ni se borran filas. PAN/NH4 quedan como ensayos originales y no se usan como objetivos de ajuste. Los valores originales y los tratados/excluidos se muestran en figuras.

Se guarda una versión nueva, separada de todos los archivos anteriores: data/cleaned/natural/lab004_009/observations_original_yan_treated_v1.csv. Se mantienen intactos el Excel original y los CSV depurados/tratados previos.

**Inicialización:** theta_full de configs/theta/theta_full.json, convertido a notación física. Dos arranques derivados SÓLO de theta_full (nominal y perturbación lognormal sigma0.2, semilla20261029), sin usar multistart07 ni el ajuste actual para arrancar. Los parámetros fijos también proceden de theta_full; los adicionales de transportadores parten de extra_initial de la configuración original. El Yef inicial de theta_full supera0.5114, por lo que se proyecta al límite factible antes de optimizar y se muestra esa conversión. No se relajan restricciones.

**Leyendas:** azul Referencia · transportadores de04; verde Nueva recalibración · datos originales + YAN tratado. La referencia se evalúa sobre los mismos datos originales para comparar.
Química: sólo LAB004–009, mismos14 parámetros libres/cotas/escalas del modelo original con transportadores, max_nfev100 por arranque. No capacidad inicial ni pérdida funcional. Después sólo kLa/gain se ajustan con CO2 LAB019–021 y química fija. Química LAB019–021 reservada; gas LAB016–018 holdout. Validación sin tratamiento de muestras. Temperaturas de logs y horarios nutricionales se conservan de la revisión de entradas, porque no son exclusiones de observaciones. LAB017 un pulso0.28g/L en ambos modelos.
Solubilidad provisional, volumen, Cd0 y objetivo RAW por medianas/igual peso por LAB son los del comparador04. Cd0=.907/.730/.829g/L en LAB019/020/021 (021 provisional), cero en LAB016–018.

Protección numérica20000 evaluaciones RHS por integración: candidato excesivamente costoso penalizado como fallo numérico; no cambia ecuaciones/tolerancias y el RHS original se restaura en memoria. Se informa su uso. No se sobreescriben módulos, ajustes ni datos anteriores. Resultados nuevos completos en JSON dentro del EXECUTED; ninguna dependencia de ejecución del repositorio antiguo. No commit/push.


In [ ]:
from pathlib import Path
import sys,json,hashlib,time
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from dataclasses import replace
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
from IPython.display import display,Markdown
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/wine_fermentation').is_dir())
sys.path.insert(0,str(ROOT/'src'));sys.dont_write_bytecode=True
assert Path(sys.prefix).name.lower()=='fermentation'
from wine_fermentation.models.with_transporters import kinetics as original
from wine_fermentation.shared.simulation import simulate as old_simulate
from wine_fermentation.shared.gas import simulate as old_gas
from wine_fermentation.shared.solubility import saturation
from wine_fermentation import model as current
from wine_fermentation.data import natural_batches
from wine_fermentation.models.historical.calibration import load
from wine_fermentation.comparison import observations,chemical_observations,CHEMICAL_COLUMNS
protected=[p for folder in ['src','configs','notebooks','results'] for p in (ROOT/folder).rglob('*') if p.is_file() and '__pycache__' not in str(p) and not p.name.startswith('08_transporter_legacy_data_treatment.')]
before={p:hashlib.sha256(p.read_bytes()).hexdigest() for p in protected}
p0=json.loads((ROOT/'configs/models/with_transporters/fit.json').read_text())['parameters']
config=json.loads((ROOT/'configs/models/with_transporters/calibration.json').read_text())
gs=json.loads((ROOT/'configs/models/with_transporters/gas_fit.json').read_text())['experiments']['train_019_021']
g0=np.array([gs['kLa'],gs['gain']])
data,training,profiles=natural_batches(ROOT);v=load(ROOT)
LABS=['LAB019','LAB020','LAB021','LAB016','LAB017','LAB018'];TRAIN=LABS[:3]
raw=pd.read_csv(ROOT/'data/processed/co2_mass_balance_lab016_021_g_l_h.csv',usecols=['lab','time_h','record_kind','remaining_volume_L','co2_raw_g_l_h'])
obs_gas=observations(raw);evaluation={};volumes={}
for lab in LABS:
 ev=raw.loc[raw.lab.eq(lab)&raw.record_kind.eq('withdrawal_after')].sort_values('time_h');et=ev.time_h.to_numpy();evv=ev.remaining_volume_L.to_numpy()
 def volume(t,et=et,evv=evv):return np.r_[2.,evv][np.searchsorted(et,np.asarray(t,float),side='right')]
 volumes[lab]=volume;b=v.batches[lab]
 if lab in TRAIN:b=replace(b,pulses={**b.pulses,'N':tuple((t,d*2/float(volume(t))) for t,d in b.pulses['N'])})
 evaluation[lab]=b

CASES=['Referencia · transportadores de 04','Nueva recalibración · datos originales + YAN tratado']
COLORS={CASES[0]:'#0072B2',CASES[1]:'#009E73'}
original_evaluation=dict(evaluation)
old17=evaluation['LAB017'];events17=old17.pulses['N']
assert np.isclose(sum(d for t,d in events17),.28)
evaluation['LAB017']=replace(old17,pulses={**old17.pulses,'N':((min(t for t,d in events17),.28),)})
print('LAB017: un pulso0.28g/L a',evaluation['LAB017'].pulses['N'][0][0],'h; aplicado a ambos modelos.')

from typing import Iterable
SECONDARY_METABOLITE_COLUMNS = {
    "PAN_mg_l": ["PAN", "Y15 PAN"],
    "NH4_mg_l": ["AMMONIA", "Y15 Amonio"],
    "pyruvic_acid": ["PYRUVIC ACID", "PYRUVIC_ACID", "Y15 Piruvic"],
    "pyruvic_acid_2": ["PYRUVIC ACID2"],
    "acetaldehyde": ["ACETALDEHIDO", "Y15 Acetaldehido"],
    "acetic_acid": ["Y15 Acetico", "ACETIC ACID", "ACETIC_ACID"],
}

AROMA_COLUMNS = {
    "benzyl_alcohol_total": ["bencil_alcohol_total", "benzyl_alcohol_total"],
    "benzaldehyde_total": ["benzaldehido_total", "benzaldehyde_total"],
    "ethyl_decanoate_total": ["decanoato_de_etilo_total", "ethyl_decanoate_total"],
    "hexyl_acetate_total": ["hexil_acetate_total", "hexyl_acetate_total"],
    "isoamyl_acetate_total": ["isoamil_acetate_total", "isoamyl_acetate_total"],
    "ethyl_octanoate_total": ["octanoate_de_etilo_total", "ethyl_octanoate_total"],
    "phenylethyl_acetate_total": ["phenylethylacetate_total", "phenylethyl_acetate_total"],
    "ethyl_acetate_total": ["Ethyl_Acetate_total", "ethyl_acetate_total"],
}

ETHANOL_DENSITY_G_ML = 0.78924

ETHANOL_G_L_PER_PERCENT_VV = 10.0 * ETHANOL_DENSITY_G_ML

CELL_MASS_PG_PER_CELL = 30.0

CELL_MASS_G_PER_CELL = CELL_MASS_PG_PER_CELL * 1e-12

MILLION_CELLS_ML_TO_KG_M3 = CELL_MASS_PG_PER_CELL * 1e-3

MG_L_TO_KG_M3 = 1e-3

def _clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(col).strip() for col in df.columns]
    return df

def _numeric(df: pd.DataFrame, column: str) -> pd.Series:
    if column not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype=float)
    return pd.to_numeric(df[column], errors="coerce")

def _first_available_numeric(df: pd.DataFrame, columns: Iterable[str]) -> pd.Series:
    values = pd.Series(np.nan, index=df.index, dtype=float)
    for column in columns:
        if column not in df.columns:
            continue
        candidate = pd.to_numeric(df[column], errors="coerce")
        values = values.where(values.notna(), candidate)
    return values

def _ethanol_observation(raw: pd.DataFrame) -> tuple[pd.Series, pd.Series, pd.Series, pd.Series]:
    primary = _numeric(raw, "ETANOL")
    primary = primary.where(primary >= 0.0)

    fallback = _first_available_numeric(raw, ["Cf Alcolyzer Real (% v/v)", "Cf Alcolyzer (% v/v)"])
    fallback = fallback.where(fallback > 0.0)

    primary_values = primary.dropna()
    primary_is_g_l = bool(primary_values.gt(25.0).any())

    percent_vv = primary.where(primary.notna(), fallback)
    e_g_l = percent_vv * ETHANOL_G_L_PER_PERCENT_VV
    unit = pd.Series("", index=raw.index, dtype=object)
    source = pd.Series("", index=raw.index, dtype=object)

    if primary_is_g_l:
        e_g_l = primary.where(primary.notna(), fallback * ETHANOL_G_L_PER_PERCENT_VV)
        percent_vv = (primary / ETHANOL_G_L_PER_PERCENT_VV).where(
            primary.notna(),
            fallback,
        )
        unit = unit.mask(primary.notna(), "g_l")
    else:
        unit = unit.mask(primary.notna(), "percent_vv")

    unit = unit.mask(primary.isna() & fallback.notna(), "percent_vv")
    source = source.mask(primary.notna(), "ETANOL")
    source = source.mask(primary.isna() & fallback.notna(), "ALCOLYZER_FALLBACK")
    return percent_vv, e_g_l, source, unit

def normalize_must_sheet(raw: pd.DataFrame, medium: str, batch: str, source_file: Path) -> pd.DataFrame:
    raw = _clean_columns(raw)
    out = pd.DataFrame(index=raw.index)
    out["medium"] = str(medium)
    out["batch"] = str(batch)
    out["source_file"] = str(source_file.name)
    out["sample_id"] = raw["ID"].astype(str) if "ID" in raw.columns else ""

    out["time_h"] = _first_available_numeric(raw, ["time", "t", "Horas"])
    out["temperature_c"] = _first_available_numeric(
        raw,
        ["temperature", "temperatura", "Temperatura_operacional_columna_Temperatura", "Temperatura"],
    )
    out["density"] = _first_available_numeric(raw, ["densidad", "Densidad"])
    out["brix"] = _first_available_numeric(raw, ["Brix"])
    out["DO_mg_l"] = _first_available_numeric(
        raw,
        ["DO", "Dissolved Oxygen", "Dissolved oxygen", "Oxigeno disuelto", "Oxígeno disuelto"],
    )
    out.loc[out["DO_mg_l"] < 0.0, "DO_mg_l"] = np.nan

    out["X_viable_mcells_ml"] = _first_available_numeric(raw, ["C_viable", "Viability"])
    out["X_total_mcells_ml"] = _first_available_numeric(raw, ["Oculyze Concentration", "C_total"])
    viable_for_dead = out["X_viable_mcells_ml"].interpolate(limit_direction="both")
    out["X_dead_direct_mcells_ml"] = (out["X_total_mcells_ml"] - out["X_viable_mcells_ml"]).clip(lower=0.0)
    out["X_dead_mcells_ml"] = (out["X_total_mcells_ml"] - viable_for_dead).clip(lower=0.0)
    out["X_dead_inferred"] = (
        out["X_total_mcells_ml"].notna()
        & out["X_viable_mcells_ml"].isna()
        & viable_for_dead.notna()
    )

    out["X_viable_kg_m3"] = out["X_viable_mcells_ml"] * MILLION_CELLS_ML_TO_KG_M3
    out["X_total_kg_m3"] = out["X_total_mcells_ml"] * MILLION_CELLS_ML_TO_KG_M3
    out["X_dead_kg_m3"] = out["X_dead_mcells_ml"] * MILLION_CELLS_ML_TO_KG_M3

    out["G_g_l"] = _first_available_numeric(raw, ["GLUCOSE", "GLUCOSE-320"])
    out["F_g_l"] = _first_available_numeric(raw, ["FRUCTOSE", "FRUCTOSE_inferida"])
    out["S_GF_g_l"] = out["G_g_l"] + out["F_g_l"]
    out.loc[out["G_g_l"].isna() | out["F_g_l"].isna(), "S_GF_g_l"] = np.nan
    out["S_Y15_g_l"] = _first_available_numeric(raw, ["Y15 Glucosa-Fructosa", "GLU-FRU-320"])
    out["S_observed_g_l"] = out["S_GF_g_l"].where(out["S_GF_g_l"].notna(), out["S_Y15_g_l"])

    out["YAN_mg_l"] = _first_available_numeric(raw, ["YAN"])
    out["N_kg_m3"] = out["YAN_mg_l"] * MG_L_TO_KG_M3
    out["N_pulse_mg_l"] = _first_available_numeric(raw, ["pulso_nut"]).fillna(0.0)
    out["N_pulse_kg_m3"] = out["N_pulse_mg_l"] * MG_L_TO_KG_M3
    out["PAN_mg_l"] = _first_available_numeric(raw, SECONDARY_METABOLITE_COLUMNS["PAN_mg_l"])
    out["NH4_mg_l"] = _first_available_numeric(raw, SECONDARY_METABOLITE_COLUMNS["NH4_mg_l"])
    out["PAN_kg_m3"] = out["PAN_mg_l"] * MG_L_TO_KG_M3
    out["NH4_kg_m3"] = out["NH4_mg_l"] * MG_L_TO_KG_M3
    out["YAN_components_mg_l"] = out["PAN_mg_l"] + out["NH4_mg_l"]
    out.loc[out["PAN_mg_l"].isna() | out["NH4_mg_l"].isna(), "YAN_components_mg_l"] = np.nan

    for normalized_column, raw_columns in SECONDARY_METABOLITE_COLUMNS.items():
        if normalized_column in {"PAN_mg_l", "NH4_mg_l"}:
            continue
        out[normalized_column] = _first_available_numeric(raw, raw_columns)

    out["glycerol_g_l"] = _first_available_numeric(raw, ["GLYCEROL"])
    (
        out["ethanol_percent_vv"],
        out["E_g_l"],
        out["ethanol_source"],
        out["ethanol_unit"],
    ) = _ethanol_observation(raw)

    for normalized_column, raw_columns in AROMA_COLUMNS.items():
        out[normalized_column] = _first_available_numeric(raw, raw_columns)

    for column in ["Condicion", "Condición", "Replica", "Réplica", "Variedad", "Levadura", "Adicion"]:
        if column in raw.columns:
            safe_column = (
                column.replace("ó", "o")
                .replace("é", "e")
                .replace("í", "i")
                .replace("á", "a")
                .replace("ú", "u")
            )
            out[safe_column] = raw[column]

    out = out.dropna(subset=["time_h"]).drop_duplicates(["medium", "batch", "time_h"], keep="first")
    out = out.sort_values("time_h").reset_index(drop=True)
    return out
SOURCE_BOOK=ROOT/'data/original/natural/lab004_012/mosto_natural_original.xlsx'
book_hash=hashlib.sha256(SOURCE_BOOK.read_bytes()).hexdigest()
previous_csv=ROOT/'data/cleaned/natural/lab004_009/observations_legacy_treatment_v1.csv'
previous_csv_hash=hashlib.sha256(previous_csv.read_bytes()).hexdigest()
raw_data=pd.concat([normalize_must_sheet(pd.read_excel(SOURCE_BOOK,sheet_name=f'LAB{i:03}'),
 'natural',f'LAB{i:03}',SOURCE_BOOK) for i in range(4,10)],ignore_index=True)
data=raw_data.copy(deep=True)
raw_batches={b.batch:b for b in current.make_batches(raw_data)}
# Keep log-temperature integration grid but replace every observation and initial from the ORIGINAL workbook.
raw_training=[]
for template in training:
 rb=raw_batches[template.batch]
 obs={}
 for state in current.STATE_NAMES:
  arr=np.full(len(template.time),np.nan)
  idx=np.searchsorted(template.time,rb.time)
  assert np.allclose(template.time[idx],rb.time)
  arr[idx]=rb.observations[state];obs[state]=arr
 raw_training.append(replace(template,observations=obs,initials=dict(rb.initials)))
training=raw_training
start_full=current.normalize_parameters(json.loads((ROOT/'configs/theta/theta_full.json').read_text()))
start_full.update(config['extra_initial'])
display(pd.Series(start_full,name='theta_full físico + adicionales de transportadores'))
display(raw_data.groupby('batch')[['X_viable_kg_m3','N_kg_m3','G_g_l','F_g_l','E_g_l']].count())


In [ ]:
SOURCE_DATA=SOURCE_BOOK
DATA_VERSION=ROOT/'data/cleaned/natural/lab004_009/observations_original_yan_treated_v1.csv'
source_hash=hashlib.sha256(SOURCE_DATA.read_bytes()).hexdigest()
original_data=raw_data.copy(deep=True);treated=original_data.copy(deep=True)
rules=[];audit=[]
for b in training:
 lab=b.batch;m=treated.batch.eq(lab)
 low=m&treated.YAN_mg_l.le(20)&treated.YAN_mg_l.notna()
 treated.loc[low,['YAN_mg_l','N_kg_m3']]=0.
 if lab=='LAB006':
  special=m&np.isclose(treated.time_h,122.)
  assert special.sum()==1 and np.isclose(original_data.loc[special,'YAN_mg_l'].iloc[0],27.)
  treated.loc[special,['YAN_mg_l','N_kg_m3']]=0.;depletion=122.
 else:
  last=max(t for t,d in b.pulses['N'])
  candidates=treated.loc[m&treated.time_h.ge(last)&treated.YAN_mg_l.eq(0),'time_h']
  assert len(candidates)>0
  depletion=float(candidates.min())
 ncut=depletion+12;xcut=float('inf')
 nlate=m&treated.time_h.gt(ncut);xlate=m&treated.time_h.gt(xcut)
 rules.append(dict(LAB=lab,agotamiento_h=depletion,corte_YAN_h=ncut,corte_X_h=xcut,
  YAN_ceros=int(low.sum())+(1 if lab=='LAB006' else 0),
  YAN_excluidos=int((nlate&treated.YAN_mg_l.notna()).sum()),
  X_excluidos=int((xlate&treated.X_viable_kg_m3.notna()).sum())))
 treated.loc[nlate,['YAN_mg_l','N_kg_m3']]=np.nan
 treated.loc[xlate,['X_viable_mcells_ml','X_viable_kg_m3']]=np.nan
for col in ['YAN_mg_l','X_viable_kg_m3']:
 for i in original_data.index:
  a=original_data.loc[i,col];z=treated.loc[i,col]
  if (pd.isna(a) and pd.isna(z)) or (pd.notna(a) and pd.notna(z) and a==z):continue
  audit.append(dict(LAB=treated.loc[i,'batch'],muestra=treated.loc[i,'sample_id'],t_h=treated.loc[i,'time_h'],variable=col,original=a,nuevo=z,accion='Excluido del ajuste' if pd.isna(z) else 'Sustituido por cero'))
allowed=['YAN_mg_l','N_kg_m3']
pd.testing.assert_frame_equal(original_data.drop(columns=allowed),treated.drop(columns=allowed))
assert len(treated)==len(original_data)
assert np.allclose(treated.YAN_mg_l/1000,treated.N_kg_m3,equal_nan=True)
DATA_VERSION.parent.mkdir(parents=True,exist_ok=True)
if DATA_VERSION.exists():pd.testing.assert_frame_equal(pd.read_csv(DATA_VERSION),treated.replace('',np.nan),check_dtype=False)
else:treated.to_csv(DATA_VERSION,index=False)
version_hash=hashlib.sha256(DATA_VERSION.read_bytes()).hexdigest()
# Re-read actual saved version, then use it for fitting.
treated=pd.read_csv(DATA_VERSION)
training_treated=[]
for b in training:
 q=treated.loc[treated.batch.eq(b.batch)]
 observations_new={}
 for state,col in current.STATE_COLUMNS.items():
  arr=np.full(len(b.time),np.nan)
  for row in q.itertuples():
   idx=np.searchsorted(b.time,row.time_h);assert b.time[idx]==row.time_h
   arr[idx]=getattr(row,col)
  observations_new[state]=arr
 training_treated.append(replace(b,observations=observations_new))
display(pd.DataFrame(rules));display(pd.DataFrame(audit))
display(pd.DataFrame([dict(LAB=b.batch,**{state:int(np.isfinite(b.observations[state]).sum()) for state in current.STATE_NAMES}) for b in training_treated]))
assert hashlib.sha256(SOURCE_DATA.read_bytes()).hexdigest()==source_hash


In [ ]:
from wine_fermentation.shared.simulation import simulate as original_simulate
integration_work={'rejected':0,'largest_rhs':0}
def simulate(b,p):
 saved_rhs=original.rhs;calls=0
 def guarded_rhs(t,y,pp,bb):
  nonlocal calls
  calls+=1
  if calls>20000:
   integration_work['rejected']+=1
   raise RuntimeError('Numerical work limit: >20000 RHS evaluations')
  return saved_rhs(t,y,pp,bb)
 original.rhs=guarded_rhs
 try:return original_simulate('with_transporters',b,p)
 finally:
  original.rhs=saved_rhs
  integration_work['largest_rhs']=max(integration_work['largest_rhs'],calls)
def chemical_residual(p,batches):
 pred={b.batch:simulate(b,p) for b in batches};parts=[]
 for b in batches:
  for state in current.STATE_NAMES:
   z=b.observations[state];m=np.isfinite(z)&(b.time>0)
   parts.append((pred[b.batch][state].to_numpy()[m]-z[m])/current.sigma_for_state(state,z[m]))
 return np.concatenate(parts)
bounds=config['bounds'];names=list(bounds);lo,hi=np.array(list(bounds.values())).T
template=dict(start_full)
rng=np.random.default_rng(20261029)
perturbed=dict(start_full)
for k,factor in zip(names,np.exp(rng.normal(0.,.2,len(names)))):perturbed[k]*=factor
seeds=[dict(start_full),perturbed]
display(pd.DataFrame([dict(parametro=k,nominal=start_full[k],perturbado=perturbed[k],
    nominal_factible=float(np.clip(start_full[k],lo[i]+1e-10,hi[i]-1e-10)),
    perturbado_factible=float(np.clip(perturbed[k],lo[i]+1e-10,hi[i]-1e-10)))
    for i,k in enumerate(names)]))

runs=[];runrows=[]
progress={'calls':0,'best':float('inf')}
def objective(logp):
 pp={**template,**dict(zip(names,np.exp(logp)))}
 try:
  r=chemical_residual(pp,training_treated);progress['calls']+=1
  progress['best']=min(progress['best'],float(np.mean(r*r)))
  if progress['calls']%25==0:print('Evaluaciones',progress['calls'],'mejor objetivo',progress['best'],'integraciones rechazadas',integration_work['rejected'],flush=True)
  return r
 except (RuntimeError,ValueError,FloatingPointError):
  count=sum(np.count_nonzero(np.isfinite(b.observations[st])&(b.time>0)) for b in training_treated for st in current.STATE_NAMES)
  return np.full(count,1e4)
for j,seed in enumerate(seeds):
 sol=least_squares(objective,np.log(np.clip([seed[k] for k in names],lo+1e-10,hi-1e-10)),bounds=(np.log(lo),np.log(hi)),max_nfev=100,diff_step=1e-3,ftol=2e-5,xtol=2e-5,gtol=2e-5)
 runs.append(sol);runrows.append(dict(inicio=j,success=bool(sol.success),objetivo=float(np.mean(sol.fun**2)),nfev=sol.nfev,message=sol.message))
 print('Calibración química inicio',j,runrows[-1],flush=True)
best=min(runs,key=lambda s:s.fun@s.fun)
pnew={**template,**dict(zip(names,np.exp(best.x)))}
assert .1<=pnew['Yeg']<=.5114 and .1<=pnew['Yef']<=.5114
singular=np.linalg.svd(best.jac,compute_uv=False)
near=[k for k,x,l,h in zip(names,np.exp(best.x),lo,hi) if min(abs(np.log(x/l)),abs(np.log(h/x)))<.02]
display(pd.DataFrame(runrows))
display(pd.DataFrame([dict(parametro=k,actual=p0[k],nuevo=pnew[k],minimo=bounds[k][0],maximo=bounds[k][1]) for k in names]))
print('Parámetros cerca de límites:',near,'; razón singular mínima/máxima:',singular[-1]/singular[0])
chemical_scores=[]
for name,pp in [(CASES[0],p0),(CASES[1],pnew)]:
 for label,batches in [('Muestras originales completas',training),('Soporte tratado común',training_treated)]:
  r=chemical_residual(pp,batches);chemical_scores.append(dict(caso=name,soporte=label,n=len(r),MSE_normalizado=float(np.mean(r*r))))
display(pd.DataFrame(chemical_scores))

print('PARAMETROS_QUIMICOS_PRECISION_COMPLETA')
print(json.dumps(pnew,indent=2,ensure_ascii=False))
print('Parámetros fijos de theta_full:')
display(pd.DataFrame([dict(parametro=k,valor=start_full[k],referencia_actual=p0[k]) for k in start_full if k not in names]))
for k in start_full:
 if k not in names:assert pnew[k]==start_full[k]


In [ ]:
def gas_cache(b,p,f,case):
 t=np.unique(np.r_[np.arange(0.,f.index[-1],.1),f.index[-1],f.index])
 yy=np.column_stack([np.interp(t,f.index,f[k]) for k in original.STATES]);rate=[]
 for ti,yi in zip(t,yy):
  q=original.terms(ti,yi,p,b);rate.append((q['mu']/12.72+(q['beta_g']+q['beta_f'])/1.05)*yi[0])
 return t,np.array(rate),saturation(np.interp(t,b.time,b.temperature_c),yy[:,6],yy[:,4],yy[:,5])
def gas_forward(cache,values,lab):
 t,rate,sat=cache;k,gain=values;cd=np.zeros(len(t));cd[0]={'LAB019':.907,'LAB020':.730,'LAB021':.829}.get(lab,0.)
 out=np.zeros(len(t));out[0]=k*max(cd[0]-sat[0],0)
 for i,dt in enumerate(np.diff(t),1):
  inventory=cd[i-1]+dt*rate[i-1];cd[i]=inventory if inventory<=sat[i] else (inventory+dt*k*sat[i])/(1+dt*k);out[i]=(inventory-cd[i])/dt
 assert np.max(abs(np.diff(cd)-np.diff(t)*(rate[:-1]-out[1:])))<1e-9
 return pd.DataFrame(dict(qpred=gain*out,Cd=cd,qprod=rate),index=t)

parameters={CASES[0]:p0,CASES[1]:pnew}
chemical={};gases={};gasfits=[];training_curves={}
for case,p in parameters.items():
 caches={}
 for b in training:training_curves[case,b.batch]=simulate(b,p)
 for lab,b in evaluation.items():
  f=simulate(b,p);chemical[case,lab]=f;caches[lab]=gas_cache(b,p,f,case)
 if case==CASES[0]:values=g0.copy();success=gs['success']
 else:
  def gres(logv):
   parts=[]
   for lab in TRAIN:
    f=gas_forward(caches[lab],np.exp(logv),lab);o=obs_gas[lab];o=o[o.time_h.between(f.index[0],f.index[-1])]
    parts.append((np.interp(o.time_h,f.index,f.qpred)-o.co2_raw_g_l_h.to_numpy())/np.sqrt(len(o)))
   return np.concatenate(parts)
  gasruns=[least_squares(gres,np.log(seed),bounds=(np.log([.01,.05]),np.log([20.,3.])),max_nfev=100,ftol=1e-7,xtol=1e-7,gtol=1e-7) for seed in (g0,[1.,.7])]
  winner=min(gasruns,key=lambda s:s.fun@s.fun);values=np.exp(winner.x);success=bool(winner.success)
 gasfits.append(dict(caso=case,kLa=values[0],gain=values[1],success=success,cerca_limite_kLa=bool(values[0]>19.6)))
 for lab in LABS:gases[case,lab]=gas_forward(caches[lab],values,lab)
display(pd.DataFrame(gasfits))
# Validate reference against original04 entry point for five unchanged input histories.
for lab in LABS:
 f=chemical[CASES[0],lab]
 reference=old_gas('with_transporters',evaluation[lab],p0,f,dict(kLa=g0[0],gain=g0[1],Cd0={'LAB019':.907,'LAB020':.730,'LAB021':.829}.get(lab,0.)),volumes[lab])
 assert np.max(abs(reference.qpred-gases[CASES[0],lab].qpred))<1e-10
# Quantify LAB017 common-input correction, without recalibration or editing source events.
old17curve=original_simulate('with_transporters',original_evaluation['LAB017'],p0)
print('LAB017 pulso único vs dos entradas antiguas: max diferencia X[g/L]=',
 float(np.max(abs(old17curve.X-chemical[CASES[0],'LAB017'].X))))

print('PARAMETROS_GAS_PRECISION_COMPLETA')
print(json.dumps(gasfits,ensure_ascii=False,indent=2))


In [ ]:
VARIABLES=[('E','Etanol',1.,'g/L'),('G','Glucosa',1.,'g/L'),('F','Fructosa',1.,'g/L'),('GF','Glucosa + fructosa',1.,'g/L'),('X','Biomasa viable',1.,'g/L'),('N','YAN',1000.,'mg/L')]
def sample(lab,key):
 if lab in TRAIN:
  o=chemical_observations(v.chem,lab,key);return o.t_h.to_numpy(),o[CHEMICAL_COLUMNS[key]].to_numpy(),'Muestras experimentales'
 if key=='GF':
  o=v.density.loc[v.density.lab.eq(lab)].dropna(subset=['t_h','density_g_l']);return o.t_h.to_numpy(),2.5*(o.density_g_l.to_numpy()-1000),'G+F por densidad'
 if key=='X':
  o=v.biomass.loc[v.biomass.lab.eq(lab)].dropna(subset=['t_h','X_viable_g_l']);return o.t_h.to_numpy(),o.X_viable_g_l.to_numpy(),'Biomasa experimental'
 return np.array([]),np.array([]),'Sin muestras disponibles'
def vals(f,key):return f.G+f.F if key=='GF' else f[key]

rows=[]
for case in CASES:
 for b in training:
  f=training_curves[case,b.batch];bt=next(x for x in training_treated if x.batch==b.batch)
  for state,title,scale,unit in VARIABLES:
   if state=='GF':continue
   for label,z in [('Original completo',b.observations[state]),('Tratado común',bt.observations[state])]:
    m=np.isfinite(z)&(b.time>0);err=(f[state].to_numpy()[m]-z[m])*scale
    if len(err):rows.append(dict(caso=case,LAB=b.batch,variable=title,rol='Calibración química',soporte=label,n=len(err),RMSE=float(np.sqrt(np.mean(err**2))),unidad=unit))
 for lab in LABS:
  f=chemical[case,lab]
  for state,title,scale,unit in VARIABLES:
   tt,zz,label=sample(lab,state);m=(tt>0)&(tt<=f.index[-1]);err=(np.interp(tt[m],f.index,vals(f,state))-zz[m])*scale
   if len(err):rows.append(dict(caso=case,LAB=lab,variable=title,rol='Holdout químico' if lab in TRAIN else 'Validación offline LAB016–018',soporte='Original completo',n=len(err),RMSE=float(np.sqrt(np.mean(err**2))),unidad=unit))
  f=gases[case,lab];o=obs_gas[lab];o=o[o.time_h.between(f.index[0],f.index[-1])];err=np.interp(o.time_h,f.index,f.qpred)-o.co2_raw_g_l_h
  rows.append(dict(caso=case,LAB=lab,variable='CO₂',rol='Calibración CO₂' if lab in TRAIN else 'Holdout CO₂',soporte='Medianas horarias RAW',n=len(err),RMSE=float(np.sqrt(np.mean(err**2))),unidad='g/(L h)'))
metrics=pd.DataFrame(rows);display(metrics.round(5))
display(metrics.groupby(['rol','soporte','variable','caso']).RMSE.mean().unstack('caso').round(5))


In [ ]:
def finish(fig,axes,title):
 handles={}
 for ax in axes.flat:
  hh,ll=ax.get_legend_handles_labels()
  for h,l in zip(hh,ll):handles.setdefault(l,h)
 fig.suptitle(title,y=.995,fontsize=14)
 fig.legend(handles.values(),handles.keys(),loc='upper center',bbox_to_anchor=(.5,.95),ncol=3,title='Modelos, muestras y tratamiento')
 fig.tight_layout(rect=(0,0,1,.83));plt.show()
# Calibración: mostrar todos los datos originales, incluso los excluidos, y los ceros impuestos.
for state,title,scale,unit in VARIABLES:
 fig,axes=plt.subplots(2,3,figsize=(15,8))
 for ax,b in zip(axes.flat,training):
  lab=b.batch
  for case in CASES:
   f=training_curves[case,lab];ax.plot(f.index,vals(f,state)*scale,color=COLORS[case],label=case)
  q=original_data.loc[original_data.batch.eq(lab)];z=treated.loc[q.index]
  col='S_GF_g_l' if state=='GF' else current.STATE_COLUMNS[state]
  orig=q[col];new=z[col];available=orig.notna();excluded=available&new.isna();modified=available&new.notna()&~np.isclose(orig,new,rtol=1e-12,atol=1e-12,equal_nan=True);kept=available&~excluded&~modified
  ax.scatter(q.loc[kept,'time_h'],orig[kept]*scale,color='black',s=22,zorder=5,label='Muestras conservadas')
  if excluded.any():ax.scatter(q.loc[excluded,'time_h'],orig[excluded]*scale,color='.6',marker='x',s=35,zorder=5,label='Muestras excluidas del ajuste')
  if modified.any():
   ax.scatter(q.loc[modified,'time_h'],orig[modified]*scale,facecolors='none',edgecolors='.5',s=32,zorder=5,label='YAN original sustituido')
   ax.scatter(q.loc[modified,'time_h'],new[modified]*scale,color='#D55E00',marker='s',s=26,zorder=6,label='YAN usado: cero')
  for j,(tp,dose) in enumerate(b.pulses['N']):ax.axvline(tp,color='.5',ls=':',lw=.8,label='Nutrición' if j==0 else None)
  if state=='N':
   cutoff=next(x['corte_YAN_h' if state=='N' else 'corte_X_h'] for x in rules if x['LAB']==lab)
   ax.axvline(cutoff,color='#CC79A7',ls='--',lw=.8,label='Corte de observaciones')
  ax.set_title(lab+' · Calibración química');ax.set_xlabel('Tiempo desde inicio [h]');ax.set_ylabel(title+' ['+unit+']');ax.grid(alpha=.25)
 finish(fig,axes,title+' · calibración LAB004–009')
# Validación química y CO2; no se alteran sus muestras.
for state,title,scale,unit in [('CO2','CO₂ emitido',1.,'g/(L h)'),*VARIABLES]:
 fig,axes=plt.subplots(2,3,figsize=(15,9))
 for ax,lab in zip(axes.flat,LABS):
  for case in CASES:
   f=gases[case,lab] if state=='CO2' else chemical[case,lab];yy=f.qpred if state=='CO2' else vals(f,state)*scale
   ax.plot(f.index,yy,color=COLORS[case],label=case)
  if state=='CO2':
   o=obs_gas[lab];ax.plot(o.time_h,o.co2_raw_g_l_h,color='black',lw=1.1,label='Medianas horarias del sensor')
  else:
   tt,zz,label=sample(lab,state);initial=tt==0
   if len(tt):
    ax.scatter(tt[~initial],zz[~initial]*scale,color='black',s=22,zorder=5,label=label)
    if initial.any():ax.scatter(tt[initial],zz[initial]*scale,facecolors='white',edgecolors='black',s=32,zorder=6,label='Muestra inicial')
   else:ax.text(.98,.95,label,ha='right',va='top',transform=ax.transAxes,fontsize=8,color='.4')
  for j,(tp,dose) in enumerate(evaluation[lab].pulses['N']):ax.axvline(tp,color='.5',ls=':',lw=.8,label='Nutrición' if j==0 else None)
  role=('Calibración CO₂' if state=='CO2' else 'Holdout químico') if lab in TRAIN else ('Holdout CO₂' if state=='CO2' else 'Validación offline')
  ax.set_title(lab+' · '+role);ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel(title+' ['+unit+']');ax.grid(alpha=.25)
 finish(fig,axes,title+' · comparación LAB019–021 y LAB016–018')


In [ ]:
display(Markdown('## Comprobaciones finales'))
assert all(hashlib.sha256(p.read_bytes()).hexdigest()==h for p,h in before.items())
assert hashlib.sha256(SOURCE_DATA.read_bytes()).hexdigest()==source_hash
assert treated.loc[treated.batch.eq('LAB006')&treated.time_h.eq(122),'YAN_mg_l'].iloc[0]==0
assert treated.loc[treated.batch.eq('LAB006')&treated.time_h.gt(134),'YAN_mg_l'].isna().all()
pd.testing.assert_series_equal(treated.X_viable_kg_m3,original_data.X_viable_kg_m3,check_names=False)
assert hashlib.sha256(SOURCE_BOOK.read_bytes()).hexdigest()==book_hash
assert hashlib.sha256(previous_csv.read_bytes()).hexdigest()==previous_csv_hash
assert pnew['Yeg']<=.5114 and pnew['Yef']<=.5114
display(metrics[metrics.variable.eq('CO₂')].groupby(['rol','caso']).RMSE.mean().unstack('caso'))
print('Todos los archivos originales protegidos permanecen intactos. Versión nueva:',DATA_VERSION.relative_to(ROOT))
print('LAB006122h=0 y corte YAN estricto134h; biomasa original conservada verificados. Muestras originales de validación intactas. Ajustes locales, sin afirmar óptimo global.')

print('Trabajo numérico:',integration_work)

assert hashlib.sha256(DATA_VERSION.read_bytes()).hexdigest()==version_hash


In [ ]:
display(Markdown('## Resultado de la nueva inicialización'))
display(metrics.groupby(['rol','soporte','variable','caso']).RMSE.mean().unstack('caso').round(6))
print('Objetivos y convergencia:',runrows)
print('Cerca de límites:',near)
print('Diferencias en mu0,Kn0,Yxn,alpha,eta0,KTr:')
display(pd.DataFrame([dict(parametro=k,referencia=p0[k],arranque_full=start_full[k],calibrado=pnew[k]) for k in ['mu0','Kn0','Yxn','alpha','eta0','KTr']]))
print('Los valores finales de YAN incluyen Nx+Ntr; no se afirma agotamiento cero sin cuantificar ambos depósitos:')
display(pd.DataFrame([dict(caso=case,LAB=lab,tiempo_final_h=float(f.index[-1]),Nx_final_mg_L=float(f.Nx.iloc[-1]*1000),Ntr_final_mg_L=float(f.Ntr.iloc[-1]*1000),YAN_final_mg_L=float(f.N.iloc[-1]*1000)) for (case,lab),f in chemical.items()]))
assert all(hashlib.sha256(p.read_bytes()).hexdigest()==h for p,h in before.items())
assert hashlib.sha256(DATA_VERSION.read_bytes()).hexdigest()==version_hash


## Lectura del ensayo

Fuente: Excel original de LAB004–009, 76 biomasas originales conservadas y sin cortes de biomasa. Sólo se trata YAN: umbral ≤20 mg/L a cero, cortes posteriores al agotamiento+12 h y LAB006 a 122 h en cero (corte estricto posterior a 134 h). Se ajustan 289 observaciones químicas posteriores al inicio. La nueva tabla se guarda aparte; los datos anteriores permanecen intactos.

Ambos arranques derivados de theta_full terminaron por xtol, con MSE normalizado 7.879782 y 7.938909. Se conserva el primero; es un ajuste local. El Yef nominal de theta_full (0.684927) se proyectó a 0.5114 antes del ajuste. Rendimientos finales Yeg=0.471047 y Yef=0.507794 g/g, ambos dentro de la restricción. Yef queda próximo al límite, sin alcanzarlo.

Frente a Referencia · transportadores de 04, en LAB019–021 el promedio de RMSE por laboratorio cambia así: YAN 39.28→35.63 mg/L y etanol 8.72→7.80 g/L; biomasa 0.557→0.586 g/L y G+F 16.17→16.39 g/L. La mejora no es uniforme. En LAB016–018, G+F por densidad cambia de 7.75 a 4.65 g/L.

La capa de CO₂ sí fue recalibrada, exclusivamente en LAB019–021 y con la nueva química fija: kLa=20 h⁻¹ (límite superior) y ganancia=0.603073. El promedio de RMSE de gas cambia de 0.163169 a 0.163750 g/(L·h) en calibración y de 0.147027 a 0.158070 en holdout LAB016–018: este último empeora. Este ensayo no demuestra una mejora global ni resuelve la identificación de transferencia.

Los valores finales de YAN al horizonte simulado se presentan separando Nx y Ntr: para el nuevo ajuste son 0.00369/0.00230/0.00073 mg/L en LAB019/020/021 y 0.01391/0.30174/0.01373 en LAB016/017/018. No se confunden con valores exactos cero ni con la última muestra offline. No hubo candidatos rechazados por el límite numérico. Modelos, configuraciones y datos previos permanecen intactos.
